In [ ]:
!pip install weaviate-client
!pip install requests
!pip install datasets

### Pobierz i przygotuj dane

Pobierz pierwsze 100 artykułów i ich streszczeń z tego datasetu: [allegro/summarization-polish-summaries-corpus](https://huggingface.co/datasets/allegro/summarization-polish-summaries-corpus/viewer/default/train?row=21)

In [ ]:
from datasets import load_dataset

data = load_dataset("allegro/summarization-polish-summaries-corpus", cache_dir="./data", split="train[:100]")

data = data.map(lambda x: {
    "pelen_tekst": x["source"],
    "streszczenie": x["target"],
}).remove_columns(["source", "target"])

### Zaimportuj klienta Weaviate

In [ ]:
import weaviate
from weaviate.classes.config import Property, DataType, Configure
from weaviate.util import generate_uuid5

### Połącz się z lokalnie działającą bazą wektorową Weaviate:

In [ ]:
client = weaviate.connect_to_local()

### Opcjonalnie usuń wszystkie dane znajdujące się w bazie

In [ ]:
client.collections.delete_all()

## Tworzenie bazy wektorów

Na początek musimy utworzyć nową kolekcję w Weaviate:
- definiujemy 2 propertiesy w kolekcji: `page_num` i `text`
- definuijemy 2 wektoryzery danych o nazwach:
    - `silver_retriever`: model [ipipan/silver-retriever-base-v1.1](https://huggingface.co/ipipan/silver-retriever-base-v1.1)
        - konfigurujemy indeks `HNSW` z włączoną kompresją `Scalar Quatization`
    - `bge_m3`: model [BAAI/bge-m3](https://huggingface.co/BAAI/bge-m3)
        - konfigurujemy indeks `flat` z włączoną kompresją `Binary Quatization`
- podpinamy generatywny moduł Ollamy z modelem: [speakleash/Bielik-11B-v2.3-Instruct](https://huggingface.co/speakleash/Bielik-11B-v2.3-Instruct)

In [ ]:
collection = client.collections.create(
        name="Articles",
        properties=[
            Property(name="pelen_tekst", data_type=DataType.TEXT),
            Property(name="streszczenie", data_type=DataType.TEXT),
        ],
        vectorizer_config=[
            Configure.NamedVectors.text2vec_transformers(
                name="silver_retriever",
                source_properties=["streszczenie"],
                inference_url="http://t2v-transformers-ipipan-silver-retriever-base-v1.1:8080",
                vectorize_collection_name=False,
                vector_index_config=Configure.VectorIndex.hnsw(
                    quantizer=Configure.VectorIndex.Quantizer.sq(),
                ),
            ),
            Configure.NamedVectors.text2vec_transformers(
                name="bge_m3",
                source_properties=["streszczenie"],
                vectorize_collection_name=False,
                vector_index_config=Configure.VectorIndex.flat(
                    quantizer=Configure.VectorIndex.Quantizer.bq(),
                ),
            ),
        ],
        generative_config=Configure.Generative.ollama(
            api_endpoint="http://generative-ollama:11434",
            model="SpeakLeash/bielik-7b-instruct-v0.1-gguf",
        ),
    )

## Importowanie danych i tworzenie embeddingów

Dzięki stworzonej konfiguracji Weaviate automatycznie zwektoryzuje dane znajdujące się w zmiennej `data` i utworzy 2 indeksy z wykorzystaniem modeli:
1. indeks: `silver_retriever` model: [ipipan/silver-retriever-base-v1.1](https://huggingface.co/ipipan/silver-retriever-base-v1.1)
2. indeks: `bge_m3` model: [BAAI/bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3)

💡 To operacja może potrwać chwilkę gdyż uruchamiamy wektoryzery lokalnie z użyciem dockera

In [ ]:
with collection.batch.dynamic() as batch:
    for obj in data:
        batch.add_object(properties=obj, uuid=generate_uuid5(obj["streszczenie"]))
    batch.flush()

Sprawdźmy czy zaimportowano wszystkie 100 obiektów

In [ ]:
count = collection.aggregate.over_all()
assert count.total_count == 100

Pobierzmy dla przykładu 3 pierwszych obiektów i zobaczmy czy posiadają one embeddingi

In [ ]:
result = collection.query.fetch_objects(limit=3, include_vector=True)

for i, obj in enumerate(result.objects):
    print(f"{i+1} obiekt\nID: {obj.uuid}\nStreszenie: {obj.properties["streszczenie"]}\nWektory dla streszczenia stworzone modelami:\n- ipipan/silver-retriever-base-v1.1 wymiar: {len(obj.vector["silver_retriever"])} wartość: {obj.vector["silver_retriever"]}\n- BAAI/bge-reranker-v2-m3 wymiar: {len(obj.vector["bge_m3"])} wartość:  {obj.vector["bge_m3"]}\n")

In [ ]:
client.close()